In [2]:

with open('data/state_of_the_union.txt', encoding='utf-8') as f:
    text_data = f.read()

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 默认分隔符 ["\n\n", "\n", " ", ""]
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=200,
                                                    chunk_overlap=20,
                                                    # 默认使用字符长度来切割 也可以使用词数来切割
                                                    length_function=len,
                                                    # 是否使用正则表达式来切割
                                                    is_separator_regex=False,
                                                    separators=[
                                                        "\n\n",
                                                        "\n",
                                                        ".",
                                                        "?",
                                                        "!",
                                                        "。",
                                                        "！",
                                                        "？",
                                                        ",",
                                                        "，",
                                                        " "
                                                    ]
                                                    )
recursive_chunks = recursive_splitter.create_documents([text_data])
print(f"分块数量：{len(recursive_chunks)}")

分块数量：10


In [9]:
import os
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='Qwen/Qwen3-Embedding-8B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [10]:
import lancedb
from langchain_community.vectorstores import LanceDB

# 创建向量数据库
db = lancedb.connect(os.path.join(os.getcwd(), "data/lancedb"))

vectorStore = LanceDB.from_documents(recursive_chunks,
                                     embedding,
                                     connection=db,
                                     table_name='state_of_the_union'
                                     )

In [12]:
vectorStore.similarity_search_with_score('长三角铁路春运')

[(Document(metadata={}, page_content='。“我来推荐我的家乡”“今年最惊喜的城市”“人少且高性价比目的地”……从不断走红的关键词和标签中可以看到，更多小众目的地正凭借口碑出圈。'),
  1.0294642448425293),
 (Document(metadata={}, page_content='与以往旅游热点扎堆不同，今年赏花春游的出游半径也在不断拉长，旅游业持续发展的当下，“春日经济”发展更加细分，为更多小众旅游地带来新的商机。据相关平台，3月中旬到4月初，以“小城市”为目的地的机票提前预定量同比增长2倍以上'),
  1.0825002193450928),
 (Document(metadata={}, page_content='草木蔓发、春山可望，春天本就自带活力和希望。主动把握消费规律，顺势而为、乘势而上，定能不断擦亮“春日经济”的底色，让文旅产业高质量发展的成色更足'),
  1.1090742349624634),
 (Document(metadata={}, page_content='“梨花风起正清明，游子寻春半出城。”最是一年春好处，赏花踏青正当时。进入3月以来，全国“赏花经济”“春日经济”热度持续攀升，全国多地迎来文旅热。不仅是传统旅游地迎来送往，小众旅游城市也正快速崛起。'),
  1.143459439277649)]